# Sinhala Letter Recognition — Transfer Learning (MobileNetV2)

Trains a letter classifier on the Kaggle Sinhala Letter dataset and exports it to **TensorFlow.js** format so it can be loaded straight into the project's Node.js Firebase Cloud Function (`functions/index.js`) via `@tensorflow/tfjs-node` — no separate Python server to run in production.

**Runtime:** Runtime → Change runtime type → GPU (T4 is fine, free tier).

**What you'll do:**
1. Pull the dataset in (Kaggle API or your own Google Drive copy)
2. Build train/validation/test splits (70/15/15 — the test set is held out and touched only once, at the end)
3. Train a MobileNetV2 transfer-learning model (frozen base → light fine-tune)
4. Evaluate properly on the held-out test set: per-class precision/recall/F1, a confusion-matrix heatmap, top-k accuracy, confidence calibration, and Grad-CAM (this is the useful bit for your thesis — which diacritics get mixed up, and why)
5. Export to TFJS + download the model + the class-name mapping

Everything downstream (the Cloud Function, the Flutter side) expects exactly the two files this notebook produces: `tfjs_model/` (a folder) and `class_names.json`.

## 1. Install dependencies

In [ ]:
!pip install -q tensorflowjs kaggle

## 2. Get the dataset

Pick **one** of the two cells below depending on where your copy of the Kaggle Sinhala Letter dataset lives. Both end with the same result: a folder `dataset/` containing one subfolder per letter class (standard `ImageFolder` layout — `dataset/<letter>/<image>.png`).

In [ ]:
# Option A — pull directly from Kaggle.
# 1. Go to kaggle.com -> your profile -> Account -> Create New API Token (downloads kaggle.json)
# 2. Run this cell, then upload kaggle.json when prompted.
from google.colab import files
import os

uploaded = files.upload()  # select your kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Replace with the exact dataset slug from its Kaggle URL, e.g. kaggle.com/datasets/<owner>/<slug>
KAGGLE_DATASET_SLUG = 'owner/sinhala-letter-dataset'

!kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p /content/raw --unzip
DATASET_DIR = '/content/raw'  # adjust if the unzip creates a nested folder

In [ ]:
# Option B — dataset already sitting in your Google Drive as a zip.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIP_PATH = '/content/drive/MyDrive/sinhala_letters.zip'  # adjust to your path
!mkdir -p /content/raw
!unzip -q "{DRIVE_ZIP_PATH}" -d /content/raw
DATASET_DIR = '/content/raw'

In [ ]:
# Sanity check — list class folders and how many images are in a few of them.
import os

class_dirs = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f'Found {len(class_dirs)} class folders')
print('First 10:', class_dirs[:10])

sample = class_dirs[0]
n_imgs = len(os.listdir(os.path.join(DATASET_DIR, sample)))
print(f'"{sample}" has {n_imgs} images')

## 3. Build train / validation / test datasets

`IMG_SIZE = 128` is a deliberate choice, not the MobileNetV2-default 224: these are single-letter line drawings, not photos, so 128×128 keeps plenty of shape detail while keeping the payload small once this ships as JSON over the wire to the Cloud Function (~49k floats vs 150k at 224).

This is a **three-way 70/15/15 split**, not a train/val-only setup. `val_ds` is used during training for early-stopping-style decisions (when to stop fine-tuning in section 6); `test_ds` is held out and evaluated exactly once, at the very end (section 7) — that's what makes the final accuracy a genuine generalization estimate rather than a number that was implicitly tuned against by watching it during training.

In [ ]:
import tensorflow as tf

IMG_SIZE = 128
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.30,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)
_val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.30,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
num_classes = len(class_names)

# Split the 30% held-out portion evenly into val (watched during training)
# and test (touched exactly once, in section 7, for the thesis numbers).
val_test_batches = tf.data.experimental.cardinality(_val_test_ds).numpy()
val_batches = val_test_batches // 2
val_ds = _val_test_ds.take(val_batches)
test_ds = _val_test_ds.skip(val_batches)

print(f'{num_classes} classes')
print(f'Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}, '
      f'Val batches: {val_batches}, Test batches: {val_test_batches - val_batches}')

In [ ]:
# Augmentation — deliberately NO horizontal/vertical flips.
# Flipping a letter is exactly the mirror-writing error this whole project is
# trying to detect elsewhere; augmenting with flips would teach the model to
# treat mirrored strokes as the same class, which is the opposite of useful.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 4. Build the model

The `Rescaling` layer bakes MobileNetV2's expected `[-1, 1]` input normalization directly into the saved model graph. That means the Cloud Function and the Flutter app only ever need to send **raw 0–255 pixel values** — normalization happens on the model side, so there's no way for training-time and serving-time preprocessing to drift out of sync (a very common source of silent accuracy loss when a model moves from Python to a different runtime).

In [ ]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(inputs)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # frozen for phase 1

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 5. Phase 1 — train the new head only

Base is frozen, so this only trains the small classification head. Fast — should take a few minutes on a free Colab GPU regardless of dataset size, since MobileNetV2 itself isn't being updated yet.

In [ ]:
EPOCHS_PHASE1 = 8

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1)

## 6. Phase 2 — light fine-tune

Unfreeze just the last ~30 layers of MobileNetV2 and train with a much lower learning rate. This squeezes out extra accuracy on the letter-specific shapes without wrecking the pretrained low-level features (edges/curves) that transfer learning is relying on. Skip this cell if phase 1 validation accuracy is already good enough — it's optional polish, not required to get a working model.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

EPOCHS_PHASE2 = 5
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE2)

## 7. Full evaluation on the held-out test set

Everything below runs on `test_ds` — the 15% the model never influenced any training or fine-tuning decision with — so these are the numbers that belong in the thesis, not `val_ds` numbers.

Covers: per-class precision/recall/F1, a confusion-matrix heatmap (screenshot-worthy — if it clusters around known diacritic-confusion pairs, that's independent evidence backing your Chapter 4 framing), top-2/top-3 accuracy (relevant since the Cloud Function already returns a top-3 list to the app), and a confidence-calibration check — the app shows the model's confidence number directly to students/teachers (the 🤖 banner in `letter_tracing_screen.dart`), so it's worth verifying whether "90% confident" actually means "right 90% of the time."

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── Collect predictions + true labels + images (images kept for Grad-CAM
# in the next cell) on the held-out test set only ──────────────────────────
y_true, y_pred, y_probs, test_images = [], [], [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_probs.extend(preds)
    test_images.extend(images.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

overall_acc = (y_true == y_pred).mean()
print(f'Test accuracy: {overall_acc:.2%}  (n={len(y_true)})')

# ── Per-class precision / recall / F1 ───────────────────────────────────────
print('\nPer-class report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

# ── Confusion matrix heatmap ────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(max(8, num_classes * 0.4), max(8, num_classes * 0.4)))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=90, colorbar=True, cmap='Blues', values_format='d'
)
ax.set_title(f'Confusion matrix — test set (accuracy {overall_acc:.1%})')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

# Text list alongside the heatmap — easier to cite specific pairs in prose.
confusions = Counter(
    (class_names[t], class_names[p]) for t, p in zip(y_true, y_pred) if t != p
)
print('\nMost confused letter pairs (true -> predicted):')
for (true_label, pred_label), count in confusions.most_common(15):
    print(f'  {true_label} -> {pred_label}: {count}')

# ── Top-k accuracy ──────────────────────────────────────────────────────────
def top_k_accuracy(y_true, y_probs, k):
    top_k_preds = np.argsort(y_probs, axis=1)[:, -k:]
    return np.mean([yt in preds for yt, preds in zip(y_true, top_k_preds)])

print()
for k in (1, 2, 3):
    print(f'Top-{k} accuracy: {top_k_accuracy(y_true, y_probs, k):.2%}')

# ── Confidence calibration: reliability diagram + Expected Calibration Error ─
confidences = y_probs.max(axis=1)
correct = (y_true == y_pred).astype(float)

n_bins = 10
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
bin_acc, bin_conf, bin_counts = [], [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (confidences > lo) & (confidences <= hi)
    if mask.sum() == 0:
        continue
    bin_acc.append(correct[mask].mean())
    bin_conf.append(confidences[mask].mean())
    bin_counts.append(mask.sum())

bin_acc, bin_conf, bin_counts = np.array(bin_acc), np.array(bin_conf), np.array(bin_counts)
ece = np.sum(bin_counts / len(confidences) * np.abs(bin_acc - bin_conf))
print(f'\nExpected Calibration Error (ECE): {ece:.4f}  (0 = perfectly calibrated)')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
ax.bar(bin_conf, bin_acc, width=1.0 / n_bins * 0.9, alpha=0.7, edgecolor='black', label='Model')
ax.set_xlabel('Predicted confidence')
ax.set_ylabel('Actual accuracy')
ax.set_title(f'Reliability diagram (ECE={ece:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('/content/calibration.png', dpi=150)
plt.show()

## 7b. Grad-CAM — which pixels actually drove a prediction

For a few examples (some correct, some misclassified), overlay a heatmap of which regions of the drawn letter the model relied on. This turns a confusion-matrix entry like "ර → ට" from *that* they're confused into *why* — e.g. the model focusing on a shared stroke shape while ignoring a distinguishing mark — which is a much stronger Chapter 4/5 artifact than the pair count alone.

In [ ]:
import matplotlib.cm as cm

# Find the base model's last conv-like (4D-output) layer automatically —
# more robust across TF versions than hardcoding a layer name like 'out_relu'.
last_conv_layer_name = None
for layer in reversed(base_model.layers):
    if len(layer.output_shape) == 4:
        last_conv_layer_name = layer.name
        break
if last_conv_layer_name is None:
    raise RuntimeError('Could not find a 4D conv layer in base_model for Grad-CAM.')

grad_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=[base_model.get_layer(last_conv_layer_name).output, model.output],
)

def make_gradcam_heatmap(img_batch, pred_index):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], (img.shape[0], img.shape[1])).numpy().squeeze()
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    jet_colors = cm.get_cmap('jet')(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    overlaid = jet_heatmap * alpha * 255 + img
    return np.clip(overlaid, 0, 255).astype(np.uint8)

# 3 correct + 3 misclassified examples from the test set.
correct_idx = np.where(y_true == y_pred)[0]
misclassified_idx = np.where(y_true != y_pred)[0]
sample_idx = list(correct_idx[:3]) + list(misclassified_idx[:3])

if sample_idx:
    fig, axes = plt.subplots(2, len(sample_idx), figsize=(3 * len(sample_idx), 6))
    if len(sample_idx) == 1:
        axes = axes.reshape(2, 1)
    for col, idx in enumerate(sample_idx):
        img = test_images[idx]
        img_batch = np.expand_dims(img, axis=0)
        heatmap = make_gradcam_heatmap(img_batch, pred_index=int(y_pred[idx]))
        overlaid = overlay_gradcam(img, heatmap)

        is_correct = y_true[idx] == y_pred[idx]
        axes[0, col].imshow(img.astype('uint8'))
        axes[0, col].axis('off')
        axes[0, col].set_title(
            f'true={class_names[y_true[idx]]}\npred={class_names[y_pred[idx]]}',
            fontsize=9, color=('green' if is_correct else 'red'),
        )
        axes[1, col].imshow(overlaid)
        axes[1, col].axis('off')

    plt.suptitle('Grad-CAM — top row: original, bottom row: overlay (left 3 correct, right 3 misclassified)')
    plt.tight_layout()
    plt.savefig('/content/gradcam_examples.png', dpi=150)
    plt.show()
else:
    print('No examples available for Grad-CAM (empty test set).')

## 8. Save class names + export to TensorFlow.js

In [ ]:
import json

with open('/content/class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print('Saved class_names.json with', len(class_names), 'entries')

In [ ]:
import tensorflowjs as tfjs

tfjs.converters.save_keras_model(model, '/content/tfjs_model')
print('Exported to /content/tfjs_model')

In [ ]:
# Zip everything and download — this is what you hand off to the app side.
!cd /content && zip -rq letter_classifier_export.zip tfjs_model class_names.json

from google.colab import files
files.download('/content/letter_classifier_export.zip')

## Next step

Unzip `letter_classifier_export.zip` and follow `training/README.md` in the project repo to plug `tfjs_model/` and `class_names.json` into the `classifySinhalaLetter` Cloud Function and deploy.